# Audio Classification with Kubeflow Trainer

This example demonstrates how to train a Convolutional Neural Network (M5 architecture) to classify music genres using the GTZAN dataset with PyTorch Distributed Data Parallel (DDP).

## Prerequisites

- FFmpeg: `apt-get install ffmpeg` or `brew install ffmpeg`
- GTZAN dataset: Download manually from [Kaggle](https://www.kaggle.com/datasets/andrewmvd/gtzan-dataset-music-genre-classification)
- Place `genres_original` folder in the same directory as this notebook

## Install Dependencies

In [9]:
!pip install -U kubeflow torch torchaudio soundfile librosa

## Define Training Function

The training function contains the M5 architecture (1D Deep CNN) for raw audio classification.

In [10]:
def train_audio_classifier():
    import os
    import glob
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader, Dataset, DistributedSampler

    # Configuration
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    
    # Initialize distributed training
    dist.init_process_group(backend=backend)
    print(f"Distributed Training for WORLD_SIZE: {dist.get_world_size()}, "
          f"RANK: {dist.get_rank()}, LOCAL_RANK: {local_rank}")
    
    device = torch.device(f"{device}:{local_rank}")

    # M5 Model Definition
    class M5(nn.Module):
        def __init__(self, n_input=1, n_output=10, stride=16, n_channel=32):
            super().__init__()
            self.conv1 = nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
            self.bn1 = nn.BatchNorm1d(n_channel)
            self.pool1 = nn.MaxPool1d(4)
            self.conv2 = nn.Conv1d(n_channel, n_channel, kernel_size=3)
            self.bn2 = nn.BatchNorm1d(n_channel)
            self.pool2 = nn.MaxPool1d(4)
            self.conv3 = nn.Conv1d(n_channel, 2 * n_channel, kernel_size=3)
            self.bn3 = nn.BatchNorm1d(2 * n_channel)
            self.pool3 = nn.MaxPool1d(4)
            self.conv4 = nn.Conv1d(2 * n_channel, 2 * n_channel, kernel_size=3)
            self.bn4 = nn.BatchNorm1d(2 * n_channel)
            self.pool4 = nn.MaxPool1d(4)
            self.fc1 = nn.Linear(2 * n_channel, n_output)

        def forward(self, x):
            x = self.conv1(x)
            x = F.relu(self.bn1(x))
            x = self.pool1(x)
            x = self.conv2(x)
            x = F.relu(self.bn2(x))
            x = self.pool2(x)
            x = self.conv3(x)
            x = F.relu(self.bn3(x))
            x = self.pool3(x)
            x = self.conv4(x)
            x = F.relu(self.bn4(x))
            x = self.pool4(x)
            x = F.avg_pool1d(x, x.shape[-1])
            x = x.permute(0, 2, 1)
            x = self.fc1(x)
            return F.log_softmax(x, dim=2).squeeze(1)

    # GTZAN Dataset
    class GTZANDataset(Dataset):
        def __init__(self, data_dir="./genres_original"):
            self.genres = ['blues', 'classical', 'country', 'disco', 'hiphop', 
                          'jazz', 'metal', 'pop', 'reggae', 'rock']
            self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}
            self.files = []
            
            for genre in self.genres:
                genre_path = os.path.join(data_dir, genre)
                if os.path.exists(genre_path):
                    wav_files = glob.glob(os.path.join(genre_path, "*.wav"))
                    self.files.extend([(f, genre) for f in wav_files])
            
            if len(self.files) == 0:
                raise RuntimeError(f"No audio files found in {data_dir}. "
                                 "Please download GTZAN dataset and place genres_original folder here.")
            
            self.target_sample_rate = 8000
            self.fixed_length = 8000 * 5  # 5 seconds

        def __len__(self):
            return len(self.files)

        def __getitem__(self, idx):
            import soundfile as sf
            import torchaudio.transforms as T
            
            filepath, genre = self.files[idx]
            data, sample_rate = sf.read(filepath, dtype='float32')
            
            waveform = torch.from_numpy(data)
            if waveform.ndim == 1:
                waveform = waveform.unsqueeze(0)
            else:
                waveform = waveform.T
            
            # Resample if needed
            if sample_rate != self.target_sample_rate:
                resampler = T.Resample(orig_freq=sample_rate, new_freq=self.target_sample_rate)
                waveform = resampler(waveform)
            
            # Pad or trim to fixed length
            if waveform.shape[1] > self.fixed_length:
                waveform = waveform[:, :self.fixed_length]
            else:
                waveform = F.pad(waveform, (0, self.fixed_length - waveform.shape[1]))
            
            return waveform, self.genre_to_idx[genre]

    # Load dataset
    dataset = GTZANDataset()
    sampler = DistributedSampler(dataset)
    train_loader = DataLoader(dataset, batch_size=4, sampler=sampler)

    # Initialize model
    model = M5(n_output=10).to(device)
    model = DDP(model, device_ids=[local_rank] if torch.cuda.is_available() else None)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.0001)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    dist.barrier()
    for epoch in range(1, 3):
        model.train()
        sampler.set_epoch(epoch)
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            if batch_idx % 10 == 0 and dist.get_rank() == 0:
                print(f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} "
                      f"({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}")

    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")
    
    dist.destroy_process_group()

## Run Locally with LocalProcessBackendConfig

In [11]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# Get torch-distributed runtime
torch_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

if torch_runtime is None:
    raise ValueError("torch-distributed runtime not found")

# Submit training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_audio_classifier,
        packages_to_install=["torch", "torchaudio", "soundfile", "librosa"],
    ),
    runtime=torch_runtime,
)

# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

T h e   R P C   c a l l   c o n t a i n s   a   h a n d l e   t h a t   d i f f e r s   f r o m   t h e   d e c l a r e d   h a n d l e   t y p e .   
 
 E r r o r   c o d e :   B a s h / S e r v i c e / 0 x 8 0 0 7 0 7 2 c 
 
 [zbf157d3453e-train] Completed with code 1 in 0:00:00.618932 seconds.T h e   R P C   c a l l   c o n t a i n s   a   h a n d l e   t h a t   d i f f e r s   f r o m   t h e   d e c l a r e d   h a n d l e   t y p e .     E r r o r   c o d e :   B a s h / S e r v i c e / 0 x 8 0 0 7 0 7 2 c   [zbf157d3453e-train] Completed with code 1 in 0:00:00.618932 seconds.

## Run on Kubernetes Cluster

**Prerequisites:** Requires a Kubernetes cluster with Kubeflow Trainer installed.

To set up a local Kind cluster:

In [13]:
!kind create cluster
!kubectl apply -k "github.com/kubeflow/training-operator/manifests/overlays/standalone"

ERROR: failed to create cluster: node(s) already exist for a cluster with the name "kind"
error: evalsymlink failure on 'C:\Users\Sneha Das\AppData\Local\Temp\kustomize-637802590\manifests\overlays\standalone' : GetFileAttributesEx C:\Users\Sneha Das\AppData\Local\Temp\kustomize-637802590\manifests\overlays\standalone: The system cannot find the file specified.


In [12]:
from kubeflow.trainer import CustomTrainer, TrainerClient

# Initialize cluster client
client = TrainerClient()

# List available runtimes
print("Available runtimes:")
torch_runtime = None
for runtime in client.list_runtimes():
    print(f"  - {runtime.name}")
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

if torch_runtime is None:
    raise ValueError("torch-distributed runtime not found")

Available runtimes:


RuntimeError: Failed to list ClusterTrainingRuntimes

In [ ]:
# Submit distributed training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_audio_classifier,
        num_nodes=2,
        resources_per_node={
            "cpu": 2,
            "memory": "4Gi",
        },
        packages_to_install=["torch", "torchaudio", "soundfile", "librosa"],
    ),
    runtime=torch_runtime,
)

In [ ]:
# Wait for running status
client.wait_for_job_status(name=job_name, status={"Running"})

# Check job steps
for step in client.get_job(name=job_name).steps:
    print(f"Step: {step.name}, Status: {step.status}, Devices: {step.device} x {step.device_count}\n")

In [ ]:
# Stream training logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

In [ ]:
# Delete job when finished
# client.delete_job(job_name)